In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/vibecoding-agents-capstone-project/NOTE.md


In [2]:
from click import prompt
import os

# Configure environment variables to use Vertex AI with the active project
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "sql-project-499120"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

import json
from datetime import date
from google.cloud import bigquery
from google.adk import Agent

def run_bigquery_query(sql_query: str) -> str:
    """Executes a GoogleSQL SELECT query against the Pakistani Banking dataset on BigQuery.
    
    Args:
        sql_query: The SQL SELECT query to execute. It must be a read-only query starting with SELECT.
        
    Returns:
        A JSON string containing the list of rows returned by the query, or an error message.
    """
    if not sql_query.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT (read-only) queries are allowed."
        
    try:
        client = bigquery.Client(project="sql-project-499120")
        query_job = client.query(sql_query)
        results = query_job.result()
        rows = [dict(row) for row in results]
        
        class CustomEncoder(json.JSONEncoder):
            def default(self, obj):
                if isinstance(obj, date):
                    return obj.isoformat()
                return super().default(obj)
                
        return json.dumps(rows, cls=CustomEncoder, indent=2)
    except Exception as e:
        return f"Error executing query: {str(e)}"

# Detailed instruction for the agent on how to write BigQuery SQL queries
INSTRUCTION = """You are a Pakistani Banking Financial Insights Agent. Your job is to answer natural language questions about Pakistani banking metrics by generating and running SQL queries on BigQuery.

You have access to a tool `run_bigquery_query` to execute SQL queries.

Dataset Information:
- Project ID: `sql-project-499120`
- Dataset ID: `Pakistan_banking` (Notice the capital 'P')

Tables and Schemas:
All tables have the exact same columns. When writing SQL, column names with spaces or special characters MUST be wrapped in backticks (e.g., `Observation Date`).
Columns:
- `Dataset Name` (STRING)
- `Observation Date` (DATE)
- `Series Key` (STRING)
- `Series Display Name` (STRING)
- `Observation Value` (FLOAT64)
- `Unit` (STRING)
- `Observation Status` (STRING)
- `Observation Status Comment` (STRING)
- `Sequence No_` (INTEGER)
- `Series name` (STRING)

Available Tables and their specific `Series name` values:

1. Table: `sql-project-499120.Pakistan_banking.banks_group_key_variables`
   Contains group-wise key variables for banks.
   Unique series names:
   - 'Total Assets of Local Private Banks'
   - 'Total Assets of Public Sector Commercial Banks'
   - 'Total Deposits of Local Private Banks'
   - 'Total Deposits of Public Sector Commercial Banks'
   - 'Total Non-Performing Loans of Local Private Banks'
   - 'Total Non-Performing Loans of Public Sector Commercial Banks'
   - 'Year-to-date Profit After Tax of Local Private  Banks' (IMPORTANT: Note that 'Local Private  Banks' has TWO spaces between 'Private' and 'Banks'!)

2. Table: `sql-project-499120.Pakistan_banking.Total_Deposits_of_Scheduled_Banks`
   Contains total deposits of scheduled banks.
   Unique series names:
   - 'Total Deposits of Schedule Banks'

3. Table: `sql-project-499120.Pakistan_banking.FSI_IslamicBanking_Features`
   Contains financial soundness indicators (FSI) for Islamic banking.
   Unique series names:
   - 'Liquid Assets ratio of Islamic Banking Institutions' (Unit: Percent)
   - 'Non-Performing Funds to Total Financing of the Islamic Banking Institutions' (Unit: Percent)
   - 'Return on Assets After Tax of the Islamic Banking Institutions' (Unit: Percent)
   - 'Return on Equity after Tax of the Islamic Banking Institutions' (Unit: Percent)

Guidelines for Query Generation:
- Always use fully qualified table names: `sql-project-499120.Pakistan_banking.<table_name>`.
- Use backticks around column names: e.g., ``SELECT `Observation Date`, `Observation Value`, `Unit` FROM ...``
- Filter by `Series name` to match the exact banking metric. For example: ``WHERE `Series name` = 'Total Deposits of Schedule Banks'``.
- Pay close attention to double spaces in 'Year-to-date Profit After Tax of Local Private  Banks' if queried.
- Ensure the query is read-only (SELECT only).
- IMPORTANT — Date Formats: Data is stored using END-OF-PERIOD dates, NOT the first day of a period.
  * Quarterly tables (banks_group_key_variables, FSI_IslamicBanking_Features) use: 2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
  * Monthly table (Total_Deposits_of_Scheduled_Banks) uses: 2025-01-31, 2025-02-28, 2025-03-31, ... 2025-08-31
  * NEVER filter with day = 01 (e.g. 2025-03-01 WILL return no data).
  * When the user says "March 2025", use: WHERE `Observation Date` = '2025-03-31'
  * When the user says "Q1 2025" or "first quarter 2025", use: WHERE `Observation Date` = '2025-03-31'
  * When the user says "2024", use: WHERE `Observation Date` BETWEEN '2024-01-01' AND '2024-12-31'
  * For "latest" or "most recent", use: ORDER BY `Observation Date` DESC LIMIT 1
- Once you receive the results of the query, explain the findings clearly to the user, highlighting key dates, values, and units (e.g. Million PKR, Percent).
"""

root_agent = Agent(
    name="pak_banking_agent",
    description="An agent to query and analyze Pakistani banking financial insights.",
    model="gemini-2.5-flash",
    instruction=INSTRUCTION,
    tools=[run_bigquery_query]
)

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


🏦 Pakistani Banking Financial Insights Agent
An AI-powered agent that answers natural language questions about Pakistan's banking sector using real State Bank of Pakistan (SBP) data stored in Google BigQuery.

Google × Kaggle 5-Day AI Agents Intensive 2026 | Agents for Business Track Submitted by: Ali Mazhar | Kaggle | LinkedIn

🎯 Problem Statement
Pakistan's State Bank publishes vast amounts of banking sector data, but it exists in raw, complex formats — hundreds of spreadsheets and tables. This creates real friction:

Bank analysts must manually dig through Excel files to extract trends
Investors and researchers need SQL knowledge just to answer basic questions
Decision-makers lose valuable time getting simple insights
Example: Finding "which bank group has the highest NPLs over the last 5 years" requires downloading multiple SBP reports, cleaning data, and writing complex queries — a process that takes hours.

💡 Solution
The Pakistani Banking Financial Insights Agent lets anyone ask questions in plain English and receive accurate, data-backed answers in seconds:

"What is the latest deposit trend for scheduled banks?"
"Which bank group has the highest non-performing loans?"
"Compare profitability between local private and public sector banks"
The agent automatically generates SQL queries, runs them against real SBP data in BigQuery, and returns clear natural language insights.

🏗️ Architecture
User (Natural Language Question)
           ↓
   pak_banking_agent
   (Google ADK 2.3 · Gemini 2.0 Flash)
           ↓
   run_bigquery_query (Agent Tool)
           ↓
   Google BigQuery
   Project: sql-project-499120
   Dataset: pakistan_banking
           ↓
   ┌─────────────────────────────────────┐
   │ banks_group_key_variables           │
   │ total_deposits_scheduled_banks      │
   │ fsi_islamic_banking                 │
   └─────────────────────────────────────┘
           ↓
   Natural Language Answer → User
📊 Data Sources
All data sourced from State Bank of Pakistan (SBP) EasyData — easydata.sbp.org.pk

Table	Description	Frequency	Period
banks_group_key_variables	Deposits, Assets, NPLs, Profit by bank group	Quarterly	2020–2025
total_deposits_scheduled_banks	Total deposits of all scheduled banks	Monthly	2020–2025
fsi_islamic_banking	Financial Soundness Indicators of Islamic Banking	Quarterly	2020–2025
🛠️ Tech Stack
Technology	Purpose
Google ADK 2.3	Agent framework
Gemini 2.0 Flash	LLM backbone
Google BigQuery	Data warehouse
Antigravity IDE	Development & testing
Agents CLI	Agent skills
Python 3.11+	Core language
google-cloud-bigquery	BigQuery Python client
🚀 Key Concepts Demonstrated
✅ Agent / ADK — Built using Google Agent Development Kit 2.3
✅ Agent Skills (Agents CLI) — Integrated via google-agents-cli
✅ Antigravity IDE — Used for development, testing, and demo
⚙️ Setup Instructions
Prerequisites
Python 3.11+
Google Cloud account with BigQuery access
Google AI Studio API key (Gemini)
uv package manager
1. Clone the repository
git clone https://github.com/alimazhar23/pakistani-banking-agent
cd pakistani-banking-agent
2. Install dependencies
pip install -r requirements.txt
3. Set up environment variables
export GOOGLE_API_KEY=your_gemini_api_key
export GOOGLE_CLOUD_PROJECT=sql-project-499120
4. Set up BigQuery
Create a Google Cloud project
Enable BigQuery API
Create dataset: pakistan_banking
Upload the 3 CSV files from SBP EasyData to BigQuery tables
5. Run the agent
python agent.py
Or run via Antigravity IDE — open the project folder and use the built-in agent runner.

💬 Example Queries
"What is the latest deposit trend for scheduled banks?"
→ Agent queries total_deposits_scheduled_banks and returns monthly trend

"Which bank group has the highest non-performing loans?"
→ Agent queries banks_group_key_variables, filters NPL series, returns comparison

"Compare profitability between local private and public sector banks"
→ Agent runs profit after tax query grouped by bank type

"How have Islamic banking soundness indicators changed since 2020?"
→ Agent queries fsi_islamic_banking and returns trend analysis
📁 Project Structure
pakistani-banking-agent/
├── agent.py              # Main agent code
├── requirements.txt      # Python dependencies
├── README.md             # This file
└── architecture.html     # Architecture diagram
⚠️ Security Note
No API keys or passwords are included in this repository. All credentials are managed via environment variables.

📈 Business Value
This agent directly addresses the "cost or revenue on the line" criteria:

Reduces analyst research time from hours to seconds
Makes SBP banking data accessible to non-technical stakeholders
Enables faster, data-driven financial decision making
Applicable to banks, regulators, investors, and financial journalists
🙏 Acknowledgements
State Bank of Pakistan for publicly available banking data via EasyData
Google & Kaggle for the AI Agents Intensive course and platform
Google ADK team for the Agent Development Kit
Submitted for Google × Kaggle 5-Day AI Agents Intensive 2026 — Agents for Business Track

